# Statistical and network analysis

This notebook reproduces the additional statistical analyses used in the revised manuscript on toxic-metal-associated genes in *Drosophila melanogaster*.

It performs two analyses:

1. **Gene-level permutation analysis of functional profiles**
   - 48 retained genes;
   - overlapping Cd-, Pb-, and Hg-associated subsets are preserved;
   - six mutually exclusive primary functional categories;
   - global Pearson-type statistic evaluated by 100,000 gene-level permutations;
   - category-specific permutation tests followed by Benjamini–Hochberg FDR correction.

2. **STRING network topology and centrality analysis**
   - 46 STRING-mapped genes;
   - minimum STRING combined interaction score = 0.400;
   - 105 retained associations;
   - degree, interaction strength, clustering coefficient, and weighted betweenness centrality;
   - weighted shortest-path distance defined as `1 / combined_score`;
   - network-level statistics are reported for the complete 46-node mapped set and for the largest connected component (LCC).

The notebook reads the frozen analytical dataset and the original STRING edge list from the repository and writes reproducible result tables to `results/`.


## Reproducibility settings

The functional categories and metal memberships are treated as **fixed input data**. The permutation procedure does not reclassify genes. A fixed random seed is used so that the empirical *P* values are exactly reproducible.

The three metal-associated subsets are not mutually exclusive. Therefore, the observed Cd/Pb/Hg membership vector of every gene is preserved during permutation; only the functional-category labels are permuted among the 48 genes.


In [ ]:
from pathlib import Path
import sys
import platform

import numpy as np
import pandas as pd
import networkx as nx

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("networkx:", nx.__version__)

B = 100_000
RANDOM_SEED = 20260919
STRING_THRESHOLD = 0.400

print(f"Permutations: {B:,}")
print("Random seed:", RANDOM_SEED)
print("STRING threshold:", STRING_THRESHOLD)


In [ ]:
# Repository paths.
# The notebook is intended to reside in code/ and to be run from either
# the repository root or the code/ directory.

cwd = Path.cwd()

if (cwd / "data" / "final_gene_set_48.csv").exists():
    ROOT = cwd
elif (cwd.parent / "data" / "final_gene_set_48.csv").exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Repository root could not be located. Expected data/final_gene_set_48.csv."
    )

GENE_FILE = ROOT / "data" / "final_gene_set_48.csv"
STRING_FILE = ROOT / "results" / "STRING_interactions_46_nodes.tsv"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

print("Repository root:", ROOT.resolve())
print("Gene dataset:", GENE_FILE)
print("STRING edge list:", STRING_FILE)


## 1. Gene-level permutation analysis

The global null hypothesis is that functional-category labels are unrelated to the observed Cd/Pb/Hg membership structure. For each permutation, the six-category labels are randomly reassigned among the 48 genes while each gene's original metal-membership vector is left unchanged.

For the global test, the statistic is

\[
T=\sum_{m,c}\frac{(O_{mc}-E_{mc})^2}{E_{mc}},
\]

where \(O_{mc}\) and \(E_{mc}\) are the observed and expected counts for metal \(m\) and functional category \(c\). The theoretical chi-square distribution is **not** used for inference. Instead, the empirical *P* value is

\[
P=\frac{b+1}{B+1},
\]

where \(B=100{,}000\) and \(b\) is the number of permutations producing a statistic at least as large as the observed value.

Because the global test is significant in the final analysis, six prespecified category-specific permutation tests are also reported. Their *P* values are adjusted using the Benjamini–Hochberg procedure.


In [ ]:
genes = pd.read_csv(GENE_FILE)

required_gene_columns = {
    "gene", "FlyBase_ID", "Cd", "Pb", "Hg",
    "primary_functional_category", "STRING_mapped", "STRING_name"
}
missing = required_gene_columns.difference(genes.columns)
if missing:
    raise ValueError(f"Missing required columns in final_gene_set_48.csv: {sorted(missing)}")

if len(genes) != 48 or genes["gene"].nunique() != 48:
    raise ValueError("Expected exactly 48 unique genes.")

METALS = ["Cd", "Pb", "Hg"]
CATEGORIES = [
    "Proteostasis",
    "Neuronal function",
    "Oxidative stress",
    "Detoxification/metal homeostasis",
    "Mitochondrial processes",
    "Signaling/other",
]

for metal in METALS:
    if not genes[metal].isin([0, 1]).all():
        raise ValueError(f"{metal} must contain only 0/1 values.")

unknown_categories = set(genes["primary_functional_category"]) - set(CATEGORIES)
if unknown_categories:
    raise ValueError(f"Unexpected functional categories: {sorted(unknown_categories)}")

print("Genes:", len(genes))
print("Metal-associated subset sizes:")
print(genes[METALS].sum().to_string())
print("\nFunctional-category counts:")
print(genes["primary_functional_category"].value_counts().reindex(CATEGORIES).to_string())


In [ ]:
# Encode the fixed metal-membership matrix (3 x 48) and category matrix (48 x 6).
M = genes[METALS].to_numpy(dtype=np.int8).T

category_codes = pd.Categorical(
    genes["primary_functional_category"],
    categories=CATEGORIES
).codes
C = np.eye(len(CATEGORIES), dtype=np.int8)[category_codes]

observed = M @ C

observed_table = pd.DataFrame(
    observed,
    index=METALS,
    columns=CATEGORIES
)

print("Observed metal × functional-category table:")
display(observed_table)


In [ ]:
def pearson_type_statistic(table):
    """Pearson-type statistic; inference is permutation-based, not chi-square-based."""
    table = np.asarray(table, dtype=float)
    expected = np.outer(table.sum(axis=1), table.sum(axis=0)) / table.sum()
    mask = expected > 0
    return float(np.sum(((table - expected) ** 2)[mask] / expected[mask]))


def category_specific_statistics(table):
    """Squared deviations of within-metal proportions from the pooled category proportion."""
    table = np.asarray(table, dtype=float)
    proportions = table / table.sum(axis=1, keepdims=True)
    pooled = table.sum(axis=0) / table.sum()
    return np.sum((proportions - pooled[None, :]) ** 2, axis=0)


def benjamini_hochberg(p_values):
    """Benjamini-Hochberg FDR-adjusted q values."""
    p = np.asarray(p_values, dtype=float)
    n = len(p)
    order = np.argsort(p)
    ranked = p[order]
    adjusted_ranked = ranked * n / np.arange(1, n + 1)
    adjusted_ranked = np.minimum.accumulate(adjusted_ranked[::-1])[::-1]
    adjusted_ranked = np.minimum(adjusted_ranked, 1.0)
    q = np.empty(n, dtype=float)
    q[order] = adjusted_ranked
    return q


T_observed = pearson_type_statistic(observed)
Tc_observed = category_specific_statistics(observed)

rng = np.random.default_rng(RANDOM_SEED)
global_exceedances = 0
category_exceedances = np.zeros(len(CATEGORIES), dtype=np.int64)

for _ in range(B):
    permuted_C = C[rng.permutation(len(genes))]
    permuted_table = M @ permuted_C

    global_exceedances += (
        pearson_type_statistic(permuted_table) >= T_observed - 1e-12
    )
    category_exceedances += (
        category_specific_statistics(permuted_table) >= Tc_observed - 1e-12
    )

global_p = (global_exceedances + 1) / (B + 1)
category_p = (category_exceedances + 1) / (B + 1)
category_q = benjamini_hochberg(category_p)

print(f"Global Pearson-type statistic: {T_observed:.6f}")
print(f"Global permutation P: {global_p:.8f}")


In [ ]:
# Prepare manuscript-oriented functional-profile results.
group_sizes = observed.sum(axis=1)

rows = []
for j, category in enumerate(CATEGORIES):
    row = {"functional_category": category}
    for i, metal in enumerate(METALS):
        n = int(observed[i, j])
        pct = 100.0 * n / group_sizes[i]
        row[f"{metal}_n"] = n
        row[f"{metal}_percent"] = pct
    row["category_statistic"] = Tc_observed[j]
    row["permutation_p"] = category_p[j]
    row["BH_FDR_q"] = category_q[j]
    rows.append(row)

functional_results = pd.DataFrame(rows)

global_results = pd.DataFrame([{
    "n_unique_genes": len(genes),
    "Cd_n": int(genes["Cd"].sum()),
    "Pb_n": int(genes["Pb"].sum()),
    "Hg_n": int(genes["Hg"].sum()),
    "permutations": B,
    "random_seed": RANDOM_SEED,
    "global_statistic": T_observed,
    "global_permutation_p": global_p,
}])

display(functional_results)
display(global_results)

# Expected values from the frozen analysis; fail loudly if the result changes.
assert np.isclose(T_observed, 23.1432, atol=1e-4)
assert np.isclose(global_p, 0.0002099979000209998, atol=1e-12)


In [ ]:
functional_results.to_csv(
    RESULTS_DIR / "functional_profile_permutation_results.csv",
    index=False
)
global_results.to_csv(
    RESULTS_DIR / "functional_profile_global_test.csv",
    index=False
)
observed_table.to_csv(
    RESULTS_DIR / "functional_profile_observed_table.csv"
)

print("Saved:")
print(RESULTS_DIR / "functional_profile_permutation_results.csv")
print(RESULTS_DIR / "functional_profile_global_test.csv")
print(RESULTS_DIR / "functional_profile_observed_table.csv")


## 2. STRING network topology and centrality

The STRING edge list is treated as an undirected functional-association network. Only associations with `combined_score >= 0.400` are accepted.

All 46 STRING-mapped genes from `final_gene_set_48.csv` are explicitly added as nodes. This retains mapped genes with no associations above the selected threshold as isolates rather than silently dropping them from the network.

For node \(i\):

- **degree** = number of incident associations;
- **strength** = sum of incident STRING `combined_score` values;
- **clustering coefficient** = unweighted local clustering coefficient;
- **weighted betweenness centrality** is calculated for the 37-node largest connected component.

For weighted shortest paths, STRING confidence is transformed to distance as

\[
d_{ij}=1/combined\_score_{ij}.
\]

This transformation ensures that stronger STRING associations correspond to shorter network distances.


In [ ]:
string_edges = pd.read_csv(STRING_FILE, sep="\t")

# STRING exports may name the first column '#node1'.
if "#node1" in string_edges.columns:
    string_edges = string_edges.rename(columns={"#node1": "node1"})

required_string_columns = {"node1", "node2", "combined_score"}
missing = required_string_columns.difference(string_edges.columns)
if missing:
    raise ValueError(f"Missing required STRING columns: {sorted(missing)}")

if (string_edges["combined_score"] < STRING_THRESHOLD).any():
    raise ValueError("STRING edge list contains interactions below the specified threshold.")

# Confirm that each undirected pair occurs only once and that there are no self-loops.
pairs = string_edges.apply(
    lambda r: tuple(sorted((str(r["node1"]), str(r["node2"])))),
    axis=1
)
if pairs.duplicated().any():
    raise ValueError("Duplicate undirected STRING edges detected.")
if (string_edges["node1"].astype(str) == string_edges["node2"].astype(str)).any():
    raise ValueError("Self-loops detected.")

mapped_nodes = (
    genes.loc[genes["STRING_mapped"].astype(int).eq(1), "STRING_name"]
    .dropna()
    .astype(str)
    .loc[lambda s: s.str.strip().ne("")]
    .tolist()
)

if len(mapped_nodes) != 46 or len(set(mapped_nodes)) != 46:
    raise ValueError("Expected exactly 46 unique STRING-mapped nodes.")

G = nx.Graph()
G.add_nodes_from(mapped_nodes)

for _, row in string_edges.iterrows():
    score = float(row["combined_score"])
    G.add_edge(
        str(row["node1"]),
        str(row["node2"]),
        combined_score=score,
        distance=1.0 / score
    )

unexpected_nodes = set(G.nodes()) - set(mapped_nodes)
if unexpected_nodes:
    raise ValueError(f"STRING edge list contains unexpected nodes: {sorted(unexpected_nodes)}")

components = sorted(nx.connected_components(G), key=len, reverse=True)
LCC_nodes = components[0]
H = G.subgraph(LCC_nodes).copy()

print("Mapped nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print("Components:", nx.number_connected_components(G))
print("Isolates:", nx.number_of_isolates(G))
print("Largest connected component:", H.number_of_nodes(), "nodes,", H.number_of_edges(), "edges")
print("combined_score range:",
      f"{string_edges['combined_score'].min():.3f}–{string_edges['combined_score'].max():.3f}")

# Structural integrity checks from the audited network.
assert G.number_of_nodes() == 46
assert G.number_of_edges() == 105
assert nx.number_connected_components(G) == 10
assert nx.number_of_isolates(G) == 9
assert H.number_of_nodes() == 37
assert H.number_of_edges() == 105


In [ ]:
degree = dict(G.degree())

strength = {
    node: sum(data["combined_score"] for _, _, data in G.edges(node, data=True))
    for node in G.nodes()
}

clustering = nx.clustering(G, weight=None)

# Weighted betweenness is calculated within the connected 37-node component.
betweenness_LCC = nx.betweenness_centrality(
    H,
    weight="distance",
    normalized=True
)

centrality = pd.DataFrame({
    "STRING_name": list(G.nodes()),
    "degree": [degree[n] for n in G.nodes()],
    "strength": [strength[n] for n in G.nodes()],
    "betweenness_weighted_LCC": [
        betweenness_LCC[n] if n in H else np.nan
        for n in G.nodes()
    ],
    "clustering_coefficient": [clustering[n] for n in G.nodes()],
    "in_largest_connected_component": [n in H for n in G.nodes()],
})

# Ranking is primarily by degree, then strength, then weighted betweenness.
centrality = centrality.sort_values(
    ["degree", "strength", "betweenness_weighted_LCC"],
    ascending=[False, False, False],
    na_position="last"
).reset_index(drop=True)
centrality.insert(0, "centrality_rank", np.arange(1, len(centrality) + 1))

display(centrality.head(15))


In [ ]:
def network_summary(graph, label):
    degrees = np.array([d for _, d in graph.degree()], dtype=float)
    return {
        "network": label,
        "nodes": graph.number_of_nodes(),
        "edges": graph.number_of_edges(),
        "density": nx.density(graph),
        "mean_degree": degrees.mean(),
        "median_degree": np.median(degrees),
        "average_clustering_unweighted": nx.average_clustering(graph, weight=None),
        "connected_components": nx.number_connected_components(graph),
        "isolates": nx.number_of_isolates(graph),
        # Retained as a secondary descriptive metric.
        "average_clustering_weighted": nx.average_clustering(
            graph, weight="combined_score"
        ),
    }

network_results = pd.DataFrame([
    network_summary(G, "Full mapped network"),
    network_summary(H, "Largest connected component"),
])

display(network_results)

print("\nTop nodes by degree:")
display(
    centrality[
        ["STRING_name", "degree", "strength", "betweenness_weighted_LCC"]
    ].head(10)
)

print("\nTop nodes by weighted betweenness within the LCC:")
display(
    centrality.loc[centrality["in_largest_connected_component"]]
    .sort_values("betweenness_weighted_LCC", ascending=False)
    [["STRING_name", "degree", "strength", "betweenness_weighted_LCC"]]
    .head(10)
)


In [ ]:
# Reproducibility checks against the finalized network analysis.
expected_degree = {
    "Hsf": 14,
    "Hsc70-5": 13,
    "Hsp83": 12,
    "Akt1": 12,
    "Sod2": 11,
    "Sod1": 10,
}

for node, expected in expected_degree.items():
    assert degree[node] == expected, f"Unexpected degree for {node}"

assert np.isclose(nx.density(G), 0.10144927536231885)
assert np.isclose(nx.average_clustering(G), 0.441080, atol=1e-6)
assert np.isclose(nx.density(H), 0.15765765765765766)
assert np.isclose(nx.average_clustering(H), 0.548369, atol=1e-6)

print("All predefined network integrity and reproducibility checks passed.")


In [ ]:
centrality.to_csv(
    RESULTS_DIR / "STRING_network_centrality_46_nodes.csv",
    index=False
)
network_results.to_csv(
    RESULTS_DIR / "STRING_network_summary.csv",
    index=False
)

print("Saved:")
print(RESULTS_DIR / "STRING_network_centrality_46_nodes.csv")
print(RESULTS_DIR / "STRING_network_summary.csv")


## Interpretation boundaries

These analyses characterize a literature-derived gene set and a STRING functional-association network. They do **not** demonstrate pathway activation, causal regulation, physical protein–protein interaction, or equivalence of toxic effects across species.

The permutation analysis tests heterogeneity in the functional composition of the curated Cd-, Pb-, and Hg-associated subsets. The network analysis quantifies topology within the selected STRING network. Centrality identifies topologically prominent nodes and should not be interpreted as direct evidence of biological indispensability or causal regulatory control.
